In [1]:
import gcsfs
import geopandas as gpd
import google.auth
import pandas as pd
import report_utils
from gtfs_curator_utils import utils
from shared_vars import PREDICTIONS_GCS, RT_MSA_DICT

credentials, _ = google.auth.default()

/home/jovyan/gtfs-curator/.venv/lib/python3.11/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.11.10) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/jovyan/gtfs-curator/.venv/lib/python3.11/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.11.10) which Google will stop supporting in new releases of google.cloud.storage_control_v2 once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.cloud.storage_control_v2 past that date.
  warnings.warn(message, FutureWarning)
/home/jovyan/gtfs-curator/.venv/lib/python3.11/site-pack

In [ ]:
# play with dropna vp_name vs without
# are these cases of left join ok in dbt model?

In [21]:
DOWNLOADED_DICT = RT_MSA_DICT.dbt_model_downloads
PROCESSED_DICT = RT_MSA_DICT.rt_schedule_models
filename = DOWNLOADED_DICT.weekday_route_direction_grain
route_df = pd.read_parquet(
    f"{PREDICTIONS_GCS}{filename}.parquet",
    filesystem=gcsfs.GCSFileSystem(),
).dropna(subset="n_tu_trips_prediction_accuracy").dropna(subset="vp_name")

In [22]:
route_df = (
    route_df.assign(
        bus_catch_likelihood=route_df[["pct_predictions_early", "pct_predictions_ontime"]].sum(axis=1),
    )
    .pipe(
        report_utils.drop_outliers,
        report_utils.MIN_ERROR_SEC_5MIN,
        report_utils.MAX_ERROR_SEC_5MIN,
    )
    .pipe(report_utils.convert_seconds_to_minutes, "avg_prediction_error_sec")
    .pipe(
        report_utils.categorize_prediction_error,
        "avg_prediction_error_minutes",
        upper_cutoff=5,
        middle_cutoff=3,
        lower_cutoff=1,
    )
    .pipe(report_utils.add_route_direction_column)
)

In [23]:
group_cols = [
    "month_first_day",
    "day_type",
    "tu_base64_url",
    "tu_name",
    "schedule_base64_url",
    "schedule_name",
    "route_dir_name",
]

# percentiles are scaled or unscaled
# For scaled, explode and get percentiles to keep, pivot
PTILES_TO_DISPLAY = [5, 25, 75]
scaled_df = report_utils.explode_percentiles(
    route_df,
    group_cols,
    array_col="scaled_prediction_error_sec_array",
    ptile_array_col="prediction_error_sec_percentile_array",
    ptiles_to_keep=[25, 75],
).rename(
    columns={
        **{f"p{i}": f"scaled_p{i}" for i in [25, 75]},
        "iqr": "scaled_iqr",
    }
)

In [24]:
route_df[group_cols].drop_duplicates().shape

(30113, 7)

In [25]:
route_df[group_cols].shape

(30113, 7)

In [26]:
route_df[route_df[group_cols].duplicated(keep=False)]

,month,year,month_first_day,day_type,schedule_base64_url,schedule_name,tu_base64_url,tu_name,vp_base64_url,vp_name,...,scaled_prediction_error_sec_array,pos_prediction_error_sec_array,neg_prediction_error_sec_array,pos_prediction_error_sec_percentile_array,n_tu_trips_prediction_accuracy,n_tu_stops_prediction_accuracy,bus_catch_likelihood,avg_prediction_error_minutes,prediction_error_label,route_dir_name
